# Aula ML 03 — Classificação

Quem cancela o plano. A acurácia da classe majoritária é o piso. O limiar e o peso da classe são a decisão.

| slide | célula |
|---|---|
| 3 | O arquivo |
| 4 | A constante |
| 5 | Probabilidade |
| 6 | Limiar |
| 7 | Matriz |
| 8 | Precisão, recall, F1 |
| 9 | Peso da classe |
| 10 | MLflow |


## De onde veio o arquivo

[Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).
A aula lê `dados/WA_Fn-UseC_-Telco-Customer-Churn.csv`.


In [ ]:
import kagglehub
kagglehub.dataset_download("blastchar/telco-customer-churn")


## ▸ Slide 3 · O arquivo

In [ ]:
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")

import mlflow
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("aula-ml-3-churn")

clientes = pd.read_csv("dados/WA_Fn-UseC_-Telco-Customer-Churn.csv")
clientes["TotalCharges"] = pd.to_numeric(clientes["TotalCharges"], errors="coerce")
print(len(clientes), "linhas · TotalCharges vazio:", int(clientes["TotalCharges"].isna().sum()))
print(clientes["Churn"].value_counts())


## ▸ Slide 4 · A constante

In [ ]:
y = (clientes["Churn"] == "Yes").astype(int)
X = clientes.drop(columns=["customerID", "Churn"])
numericas = ["tenure", "MonthlyCharges", "TotalCharges"]
categoricas = [c for c in X.columns if c not in numericas]
for c in categoricas:
    X[c] = X[c].astype(str)

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)
print("fração que fica, no arquivo:", round(float((y == 0).mean()), 3))
print("acurácia da constante no teste:", round(float((y_teste == 0).mean()), 3))

def avaliar(y_verdadeiro, probabilidade, corte):
    classe = (probabilidade >= corte).astype(int)
    return {
        "acuracia": accuracy_score(y_verdadeiro, classe),
        "precisao_sai": precision_score(y_verdadeiro, classe, zero_division=0),
        "recall_sai": recall_score(y_verdadeiro, classe, zero_division=0),
        "f1_sai": f1_score(y_verdadeiro, classe, zero_division=0),
        "alertas": int(classe.sum()),
    }

with mlflow.start_run(run_name="constante-fica"):
    notas = avaliar(y_teste, (y_teste * 0).to_numpy(), 0.5)
    mlflow.log_metrics(notas)
    print(notas)


## Preparo — o mesmo desenho da aula 2

In [ ]:
preparo = ColumnTransformer([
    ("num", Pipeline([
        ("nulos", SimpleImputer(strategy="median")),
        ("escala", StandardScaler()),
    ]), numericas),
    ("cat", Pipeline([
        ("nulos", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categoricas),
])
modelo = Pipeline([
    ("preparo", preparo),
    ("clf", LogisticRegression(max_iter=1000)),
])
modelo.fit(X_treino, y_treino)
probabilidade = modelo.predict_proba(X_teste)[:, 1]


## ▸ Slides 6 a 8 · Limiar, matriz e métricas

In [ ]:
for corte in (0.3, 0.5, 0.7):
    notas = avaliar(y_teste, probabilidade, corte)
    print(corte, {k: round(v, 3) if isinstance(v, float) else v for k, v in notas.items()})
    with mlflow.start_run(run_name=f"logistica-corte-{corte}"):
        mlflow.log_params({"corte": corte})
        mlflow.log_metrics(notas)

print(confusion_matrix(y_teste, (probabilidade >= 0.5).astype(int)))


## ▸ Slide 9 · Peso da classe

In [ ]:
balanceada = Pipeline([
    ("preparo", preparo),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
balanceada.fit(X_treino, y_treino)
notas = avaliar(y_teste, balanceada.predict_proba(X_teste)[:, 1], 0.5)
print(notas)
with mlflow.start_run(run_name="logistica-balanceada"):
    mlflow.log_params({"corte": 0.5, "class_weight": "balanced"})
    mlflow.log_metrics(notas)
    mlflow.sklearn.log_model(
        balanceada, name="modelo", serialization_format="pickle",
    )
